# Study 828 — FX Dollar Factor — the teardown

The DOL premium Newey-West *t*, the dollar-timing regression + block-shuffle placebo, the two-era cut, the costed static/timed books, and the 20-seed synthetic control.

In [1]:
R = {'start': '2003-12-31', 'end': '2026-06-30', 'n_ccy': 7, 'n_months': 270, 'fingerprint': 'de324e1d9334', 'spot_bps': 0.52, 'spot_ann': 0.06, 'spot_t_nw': 0.04, 'spot_t_1s': 0.04, 'spot_sharpe': 0.01, 'vol': 7.7, 'ex_bps': 1.24, 'ex_ann': 0.15, 'ex_t_nw': 0.09, 'tim_beta': -0.0307, 'tim_t': -1.46, 'tim_r2': 0.0101, 'tim_n': 258, 'placebo_obs': 0.0307, 'placebo_sd': 0.0238, 'placebo_p': 0.084, 'placebo_rot': 1000, 'era_early_bps': 8.91, 'era_early_ann': 1.07, 'era_early_t': 0.38, 'era_early_n': 132, 'era_late_bps': -7.5, 'era_late_ann': -0.9, 'era_late_t': -0.55, 'era_late_n': 138, 'static_gross': -0.29, 'static_net': -0.41, 'static_sharpe': -0.05, 'static_t': -0.25, 'timed_gross': -0.88, 'timed_net': -0.96, 'timed_sharpe': -0.18, 'timed_t': -0.86, 'switches': 1.63, 'invested': 51, 'null_prem_mean_t': 0.32, 'null_prem_sd': 0.84, 'null_prem_fire': 0, 'null_tim_fire': 1, 'planted_prem_bps': 24.41, 'planted_prem_t': 3.63, 'planted_tim_beta': 0.055, 'planted_tim_t': 5.23}

## The headline — the DOL premium

Equal-weight foreign-currency-vs-USD basket, monthly.

In [2]:
print(f"DOL spot   : {R['spot_bps']:+.2f} bps/mo ({R['spot_ann']:+.2f}%/yr)  "
      f"NW(6) t = {R['spot_t_nw']:+.2f}  one-sample t = {R['spot_t_1s']:+.2f}  Sharpe {R['spot_sharpe']:+.2f}")
print(f"DOL excess : {R['ex_bps']:+.2f} bps/mo ({R['ex_ann']:+.2f}%/yr)  NW(6) t = {R['ex_t_nw']:+.2f}  (+carry proxy)")
print(f"vol        : {R['vol']:.1f}%/yr")

DOL spot   : +0.52 bps/mo (+0.06%/yr)  NW(6) t = +0.04  one-sample t = +0.04  Sharpe +0.01
DOL excess : +1.24 bps/mo (+0.15%/yr)  NW(6) t = +0.09  (+carry proxy)
vol        : 7.7%/yr


## Dollar-timing — predictive regression DOL_{t+1} = a + b·signal_t

signal = trailing-12m DOL (a spot-only proxy for the average forward discount).

In [3]:
print(f"slope b   : {R['tim_beta']:+.4f}   NW(6) t = {R['tim_t']:+.2f}   R2 = {R['tim_r2']:.4f}   n = {R['tim_n']}")
print(f"placebo   : |obs b| {R['placebo_obs']:.4f} vs placebo sd {R['placebo_sd']:.4f} "
      f"({R['placebo_rot']} rotations) -> p = {R['placebo_p']:.3f}")
print('  the slope is insignificant AND wrong-signed (high trend -> lower next DOL)')

slope b   : -0.0307   NW(6) t = -1.46   R2 = 0.0101   n = 258
placebo   : |obs b| 0.0307 vs placebo sd 0.0238 (1000 rotations) -> p = 0.084
  the slope is insignificant AND wrong-signed (high trend -> lower next DOL)


## Robustness — two eras (split 2015-01-01)

In [4]:
print(f"2004-2014 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps/mo ({R['era_early_ann']:+.2f}%/yr)  NW t = {R['era_early_t']:+.2f}")
print(f"2015-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps/mo ({R['era_late_ann']:+.2f}%/yr)  NW t = {R['era_late_t']:+.2f}")
print('  the premium flips sign across the halves -> not a stable factor')

2004-2014 (n=132): +8.91 bps/mo (+1.07%/yr)  NW t = +0.38
2015-2026 (n=138): -7.50 bps/mo (-0.90%/yr)  NW t = -0.55
  the premium flips sign across the halves -> not a stable factor


## The timer — can you get paid for it?

Static long basket (20% turnover) and a timed overlay (long when trend>0), 5 bps one-way.

In [5]:
print(f"static long DOL : gross {R['static_gross']:+.2f}%/yr -> net {R['static_net']:+.2f}%/yr "
      f"(Sharpe {R['static_sharpe']:+.2f}, t {R['static_t']:+.2f})")
print(f"timed DOL       : gross {R['timed_gross']:+.2f}%/yr -> net {R['timed_net']:+.2f}%/yr "
      f"(Sharpe {R['timed_sharpe']:+.2f}, t {R['timed_t']:+.2f}, {R['switches']:.2f} switches/yr, invested {R['invested']}%)")

static long DOL : gross -0.29%/yr -> net -0.41%/yr (Sharpe -0.05, t -0.25)
timed DOL       : gross -0.88%/yr -> net -0.96%/yr (Sharpe -0.18, t -0.86, 1.63 switches/yr, invested 51%)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover planted premium + timing.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from dollar_factor import data, strategy as st
npt = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, timing=0.0, seed=828+s, n_months=480))['prem_t_nw'] for s in range(10)])
ntt = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, timing=0.0, seed=828+s, n_months=480))['timing_t'] for s in range(10)])
print(f"null (10 seeds): premium NW t mean {npt.mean():+.2f} (|t|>=2 in {(abs(npt)>=2).sum()}/10); timing |t|>=2 in {(abs(ntt)>=2).sum()}/10")
prem = st.synthetic_detect(data.synthetic_panel(edge=0.004, timing=0.0, seed=828, n_months=480))
tim  = st.synthetic_detect(data.synthetic_panel(edge=0.0, timing=0.02, seed=828, n_months=480))
print(f"planted premium: DOL {prem['prem_mean_bps']:+.2f} bps/mo, NW t = {prem['prem_t_nw']:+.2f}")
print(f"planted timing : slope {tim['timing_beta']:+.4f}, t = {tim['timing_t']:+.2f}")

null (10 seeds): premium NW t mean +0.48 (|t|>=2 in 0/10); timing |t|>=2 in 1/10
planted premium: DOL +24.41 bps/mo, NW t = +3.63
planted timing : slope +0.0549, t = +5.23


## Verdict

- **Signal — None.** The unconditional dollar factor earns **+0.06%/yr** (NW *t* = **+0.04**) — indistinguishable from zero, and LRV's own premise. The dollar-timing test with the spot-only forward-discount proxy is **insignificant and wrong-signed** (NW *t* = **-1.46**, placebo p = 0.084), and the premium flips sign across eras (*t* = +0.38 / -0.55). The 20-seed synthetic control recovers a *planted* premium (*t* = +3.63) and *planted* timing (*t* = +5.23) cleanly, firing on 0/20 premium nulls — so the flat tape is a real absence. *(The true rate-based average forward discount is not reconstructable from spot — a data limit, honestly flagged.)*
- **Tradability — Mirage.** The static long basket nets **-0.41%/yr** and the timed overlay **-0.96%/yr** at 5 bps — no premium to pay for the friction.